# Compare Cancels

把撤单对比逻辑单独拆出来，支持两种使用方式：

1. 直接使用下面新增的前置 cell，自动生成 `order_data`
2. 如果你已经在别的 notebook 里有 `order_data` / `cancels`，也可以直接复用当前内存变量

默认流程：
- 先通过 `examples.helper` 的公共逻辑生成 `order_data`
- 如有需要，用 `orderbook.export_cancels_csv("cancels.csv")` 先导出撤单文件
- 再运行最后的对比 cell


In [ ]:
import sys
from pathlib import Path

import pandas as pd
import polars as pl
from IPython.display import display

project_root = Path.cwd()
if (
    not (project_root / "examples").exists()
    and (project_root.parent / "examples").exists()
):
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from examples.helper import (  # noqa: E402
    DEFAULT_CUT_TIME,
    build_legacy_loader,
    load_symbol_frames,
    run_single_validation,
)

In [ ]:
DATE = "2026-04-03"
BROKER = "guoxin"
COLO = "SH"
SYMBOL = "600000"
MARKET = "SH"
QUOTE_DIR = "/mnt/data/private/data001/hq/lob-test/HQ/20260403sh"
BASE_ARCHIVE_DIR = "/mnt/beegfs/quant002/hds_work/DailyMd/"
CUT_TIME = DEFAULT_CUT_TIME

loader = build_legacy_loader(
    date=DATE,
    broker=BROKER,
    colo=COLO,
    base_archive_dir=BASE_ARCHIVE_DIR,
)

trade_df, order_df = load_symbol_frames(loader, QUOTE_DIR, SYMBOL)
matched, order_data, orderbook = run_single_validation(
    trade_df=trade_df,
    order_df=order_df,
    symbol=SYMBOL,
    market=MARKET,
    cut_time=CUT_TIME,
)

print(f"single check: {'OK' if matched else 'MISMATCH'}")
order_data.head()

In [ ]:
# 如果你还没有外部提供的撤单文件，可以先导出一份本地 cancels.csv 再继续对比
orderbook.export_cancels_csv("cancels.csv")

In [ ]:
def compare_cancels(order_data: pd.DataFrame, cancels: pd.DataFrame):
    """
    order_data: 处理后的逐笔合并df（pandas），撤单条件：type='O' & order_type='D'.

    cancels: 你导出的撤单df（pandas），列名如截图：
    取消ID/订单ID/方向/价格/数量/挂单时间/取消时间.

    返回：
      - od_cancel: 从 order_data 抽取的撤单事件
      - ca_cancel: cancels 规范化后的撤单事件
      - missing_in_cancels: order_data有但cancels没有
      - extra_in_cancels: cancels有但order_data没有
      - time_mismatch: 同订单ID能对上，但取消时间不同（可选诊断）
    """
    required = ["type", "order_type"]
    for c in required:
        if c not in order_data.columns:
            msg = f"order_data 缺少列: {c}, 实际列={list(order_data.columns)}"
            raise KeyError(msg)

    if "orderorino" in order_data.columns:
        oid_col = "orderorino"
    elif "order_id" in order_data.columns:
        oid_col = "order_id"
    else:
        msg = f"order_data 缺少订单ID列（orderorino/order_id），实际列={list(order_data.columns)}"
        raise KeyError(msg)

    if "time" in order_data.columns:
        cancel_time_col = "time"
    elif "int_time" in order_data.columns:
        cancel_time_col = "int_time"
    else:
        msg = (
            f"order_data 缺少时间列（time/int_time），实际列={list(order_data.columns)}"
        )
        raise KeyError(msg)

    pick_cols = ["serial"] if "serial" in order_data.columns else []
    pick_cols += [oid_col, cancel_time_col]
    if "full_time" in order_data.columns:
        pick_cols += ["full_time"]

    od_cancel = order_data.loc[
        (order_data["type"] == "O") & (order_data["order_type"] == "D"), pick_cols
    ].copy()

    od_cancel = od_cancel.rename(
        columns={
            oid_col: "order_id",
            cancel_time_col: "cancel_time",
            "full_time": "cancel_full_time",
        }
    )

    od_cancel["order_id"] = pd.to_numeric(
        od_cancel["order_id"], errors="coerce"
    ).astype("Int64")
    od_cancel["cancel_time"] = pd.to_numeric(
        od_cancel["cancel_time"], errors="coerce"
    ).astype("Int64")
    if "cancel_full_time" in od_cancel.columns:
        od_cancel["cancel_full_time"] = pd.to_numeric(
            od_cancel["cancel_full_time"], errors="coerce"
        ).astype("Int64")

    od_cancel = od_cancel.dropna(subset=["order_id", "cancel_time"]).drop_duplicates(
        subset=["order_id", "cancel_time"]
    )

    col_map = {}
    if "订单ID" in cancels.columns:
        col_map["订单ID"] = "order_id"
    if "取消时间" in cancels.columns:
        col_map["取消时间"] = "cancel_time"
    if "挂单时间" in cancels.columns:
        col_map["挂单时间"] = "order_time"
    if "取消ID" in cancels.columns:
        col_map["取消ID"] = "cancel_id"
    if "价格" in cancels.columns:
        col_map["价格"] = "price"
    if "数量" in cancels.columns:
        col_map["数量"] = "qty"
    if "方向" in cancels.columns:
        col_map["方向"] = "side"

    ca_cancel = cancels.rename(columns=col_map).copy()

    need_ca = ["order_id", "cancel_time"]
    for c in need_ca:
        if c not in ca_cancel.columns:
            msg = f"cancels 缺少列: {c}（原始列名可能不是'订单ID/取消时间'），实际列={list(cancels.columns)}"
            raise KeyError(msg)

    ca_cancel["order_id"] = pd.to_numeric(
        ca_cancel["order_id"], errors="coerce"
    ).astype("Int64")
    ca_cancel["cancel_time"] = pd.to_numeric(
        ca_cancel["cancel_time"], errors="coerce"
    ).astype("Int64")
    ca_cancel = ca_cancel.dropna(subset=["order_id", "cancel_time"]).drop_duplicates(
        subset=["order_id", "cancel_time"]
    )

    od_key = od_cancel[["order_id", "cancel_time"]].copy()
    ca_key = ca_cancel[["order_id", "cancel_time"]].copy()

    merged = od_key.merge(
        ca_key, on=["order_id", "cancel_time"], how="outer", indicator=True
    )

    missing_in_cancels = merged.loc[
        merged["_merge"] == "left_only", ["order_id", "cancel_time"]
    ].copy()
    extra_in_cancels = merged.loc[
        merged["_merge"] == "right_only", ["order_id", "cancel_time"]
    ].copy()

    od_by_oid = od_cancel[["order_id", "cancel_time"]].drop_duplicates()
    ca_by_oid = ca_cancel[["order_id", "cancel_time"]].drop_duplicates()

    time_mismatch = (
        od_by_oid.merge(ca_by_oid, on="order_id", how="inner", suffixes=("_od", "_ca"))
        .query("cancel_time_od != cancel_time_ca")
        .sort_values(["order_id", "cancel_time_od", "cancel_time_ca"])
    )

    print(f"[order_data 撤单事件数] {len(od_key)}")
    print(f"[cancels 撤单事件数] {len(ca_key)}")
    print(f"[order_data 有但 cancels 没有] {len(missing_in_cancels)}")
    print(f"[cancels 有但 order_data 没有] {len(extra_in_cancels)}")
    print(f"[同订单ID但取消时间不一致] {len(time_mismatch)}")

    return {
        "od_cancel": od_cancel.sort_values(["cancel_time", "order_id"]),
        "ca_cancel": ca_cancel.sort_values(["cancel_time", "order_id"]),
        "missing_in_cancels": missing_in_cancels.sort_values(
            ["cancel_time", "order_id"]
        ),
        "extra_in_cancels": extra_in_cancels.sort_values(["cancel_time", "order_id"]),
        "time_mismatch": time_mismatch,
    }

In [ ]:
# 如果当前环境里没有 cancels，就尝试从当前目录读取 cancels.csv
if "cancels" not in globals():
    csv_path = Path("cancels.csv")
    if not csv_path.exists():
        msg = "当前没有 cancels 变量，且目录下也没有 cancels.csv"
        raise FileNotFoundError(msg)
    cancels = pl.read_csv(csv_path)

if "order_data" not in globals():
    msg = "当前没有 order_data 变量，请先在 notebook 里准备好 order_data"
    raise NameError(msg)

if not isinstance(cancels, pd.DataFrame):
    cancels = cancels.to_pandas()

out = compare_cancels(order_data, cancels)

display(out["missing_in_cancels"].head(20))
display(out["extra_in_cancels"].head(20))
display(out["time_mismatch"].head(20))